# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIRˆ2) Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIRˆ2 dataset using the `mlcroissant` library. It demonstrates Croissant-based record set access, data processing, and visualization following best practices.

### Dataset Source
Source: [FAIR2 Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the FAIRˆ2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their IDs (`@id`), fields, and columns. All Croissant entities are referenced by their `@id`.

In [ ]:
# List all record sets available using their @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '<no name>')}")

# For demonstration, access the fields and columns for the first record set
if record_sets:
    rs = record_sets[0]
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"\nFields for record set {rs['@id']}:")
    for field in fields:
        print(f"  Field @id: {field['@id']}, name: {field.get('name', '<no name>')}, dataType: {field.get('dataType', '<unknown>')}")
    # Print column IDs if available
    columns = rs.get('column', [])
    print(f"\nColumns for record set {rs['@id']}:")
    if columns:
        if isinstance(columns, dict):
            columns = [columns]
        for col in columns:
            print(f"  Column @id: {col['@id']}, name: {col.get('name', '<no name>')}")

## 3. Data Extraction
Load records from each record set into Pandas DataFrames using each record set's `@id`. (Use the IDs discovered in the overview step.)

In [ ]:
# List of record set @ids to extract (edit as needed if multiple exist)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Print column names for the first record set
main_rs_id = record_set_ids[0]
print(f"Columns in record set {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())

# Show first 5 rows
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply best-practice processing: filter records, normalize numeric fields, group/categorize data, etc. Reference all columns and fields by their `@id` where possible.

In [ ]:
# Pick a numeric field and a grouping field by @id (adjust below to match actual @ids)
df = dataframes[main_rs_id]
# For demonstration, choose a plausible numeric field and a grouping field.

# Print columns to help user pick field @ids:
print("Columns available:")
print(df.columns.tolist())

# Example: Suppose '@id': 'cr:field:Interval_between_diagnoses_days' is numeric, '@id': 'cr:field:Sex' is categorical
# Try to use @id names when available
numeric_field = None
group_field = None

# Try to auto-detect a numeric field: take the first that looks like interval, age, or count
for col in df.columns:
    if 'interval' in col.lower() or 'age' in col.lower() or 'count' in col.lower() or 'Number' in col:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
if numeric_field is None:
    # Default to the first numeric column
    num_cols = df.select_dtypes(include=[float, int]).columns
    if len(num_cols) > 0:
        numeric_field = num_cols[0]

# For grouping, look for 'Sex', 'Location', 'MSI', or anatomical/categorical field
for col in df.columns:
    if 'sex' in col.lower() or 'anatomical' in col.lower() or 'msi' in col.lower() or 'site' in col.lower():
        if pd.api.types.is_object_dtype(df[col]):
            group_field = col
            break

print(f"\nUsing numeric field: {numeric_field}")
print(f"Using group/categorical field: {group_field}")

if numeric_field in df.columns:
    # Filtering: keep values > threshold
    threshold = df[numeric_field].quantile(0.25) if pd.notnull(df[numeric_field]).sum() > 0 else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame('mean_'+numeric_field)
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field detected in this dataset for demo.")

## 5. Visualization
Visualize data distributions and group-wise comparisons. This demo uses matplotlib; adapt visualizations according to relevant fields and dataset size.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we:
- Loaded the FAIRˆ2 dataset via Croissant schema using `mlcroissant`.
- Explored record sets, fields, and their `@id`s for full transparency and reproducibility.
- Extracted records into DataFrames, performed basic EDA (filtering, normalization, grouping), and produced visualizations of key attributes.

All identifiers link directly to their Croissant schema definitions, supporting robust and future-proof data science workflows.